In [1]:
!nvidia-smi

Sun Mar 15 16:53:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U bitsandbytes
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 51.8 MB/s eta 0:00:00


In [3]:
import os
import sys
import torch
from transformers import AutoTokenizer, GenerationConfig, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

model_name = 'LiquidAI/LFM2.5-1.2B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    #quantization_config=bnb_config, # Раскомментировать в случае использования моделей побольше, где не нужна квантизация
    device_map="cuda:0", # В случае нескольких GPU нужно указать конкретную, cuda:0, например
    attn_implementation="sdpa", #attn_implementation='flash_attention_2' для GPU, которые поддерживают
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(2048, 2048, kernel_size=(3,), stride=(1,), padding=(2,), groups=2048, bias=False)
          (in_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear(in_features=2048, out_features=8192, bias=False)
          (w3): Linear(in_features=2048, out_features=8192, bias=False)
          (w2): Linear(in_features=8192, out_features=2048, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((2048,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((2048,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Li

In [4]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(model_name)
generation_config

GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 7,
  "pad_token_id": 0
}

In [5]:
generation_config.temperature = 0.3
generation_config.max_new_tokens = 512
generation_config

GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 7,
  "max_new_tokens": 512,
  "pad_token_id": 0,
  "temperature": 0.3
}

In [6]:
!pip install datasets

In [11]:
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import re

dataset = load_dataset("RussianNLP/coat", "binary", trust_remote_code=True)
test_data = dataset['train']

test_sample = test_data.shuffle(seed=42).select(range(100))

def build_prompt(text):
    messages = [
        {"role": "user", "content": f"Определи, является ли следующий текст написанным человеком или сгенерированным искусственным интеллектом. Ответь только одним словом: 'человек' или 'ии'.\n\nТекст: {text}\n\nОтвет:"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

def get_model_prediction(text):
    prompt = build_prompt(text)
    input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated_ids = outputs[0][input_ids.shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()
    return answer

def extract_label(answer):
    if 'человек' in answer or 'human' in answer:
        return 0
    elif 'ии' in answer or 'ai' in answer or 'machine' in answer or 'generated' in answer:
        return 1
    else:
        return None

true_labels = []
pred_labels = []
texts = []

for example in test_sample:
    text = example['text']
    true_label = example['label']
    texts.append(text)
    true_labels.append(true_label)

    answer = get_model_prediction(text)
    pred = extract_label(answer)
    pred_labels.append(pred if pred is not None else -1)

    if len(true_labels) <= 5:
        print(f"Текст: {text[:100]}...")
        print(f"Ответ модели: {answer}")
        print(f"Истинная метка: {true_label}, предсказанная: {pred}")
        print("-" * 50)

valid_indices = [i for i, p in enumerate(pred_labels) if p != -1]
if len(valid_indices) < len(pred_labels):
    print(f"Не удалось распознать ответ для {len(pred_labels) - len(valid_indices)} примеров. Они исключены из оценки.")
    true_labels = [true_labels[i] for i in valid_indices]
    pred_labels = [pred_labels[i] for i in valid_indices]

accuracy = accuracy_score(true_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')

print(f"\nРезультаты zero-shot оценки на {len(true_labels)} примерах:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'RussianNLP/coat' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'RussianNLP/coat' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Текст: Вначале, роль АВС была ограничена вопросами стандартизации military equipment, training, и тактики....
Ответ модели: человек
Истинная метка: 1, предсказанная: 0
--------------------------------------------------
Текст: Он участвовал в телеконференции с президентом РФ, которая состоялась накануне. По мнению президента ...
Ответ модели: человек
Истинная метка: 1, предсказанная: 0
--------------------------------------------------
Текст: Комплексная организация отдыха и развлечений в городе Туапсе. Отдых на море, отдых по доступным цена...
Ответ модели: человек
Истинная метка: 1, предсказанная: 0
--------------------------------------------------
Текст: Органы местного самоуправления обязаны содержать, обслуживать и обеспечивать сохранность временно не...
Ответ модели: человек
Истинная метка: 0, предсказанная: 0
--------------------------------------------------
Текст: Эти тексты описывают происхождение и природу Восточной Чистой Земли, в которой находится Будда Амита...
Ответ моде

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
import numpy as np
from collections import Counter

print("Распределение истинных меток:", Counter(true_labels))
print("Распределение предсказанных меток:", Counter(pred_labels))

Распределение истинных меток: Counter({0: 54, 1: 46})
Распределение предсказанных меток: Counter({0: 100})


In [13]:
import requests
import zipfile

url = "https://russiansuperglue.com/tasks/download/PARus"
response = requests.get(url)
with open("PARus.zip", "wb") as f:
    f.write(response.content)

with zipfile.ZipFile("PARus.zip", "r") as zip_ref:
    zip_ref.extractall("PARus_data")

!ls PARus_data/

__MACOSX  PARus


In [14]:
data_files = {
    "train": "PARus_data/PARus/train.jsonl",
    "validation": "PARus_data/PARus/val.jsonl",
    "test": "PARus_data/PARus/test.jsonl"
}

dataset = load_dataset('json', data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [15]:
num_samples = 100
train_sample = dataset['train'].shuffle(seed=42).select(range(num_samples))

def build_parus_prompt(premise, choice1, choice2, question):
    if question == "cause":
        q_text = "причиной"
    else:
        q_text = "следствием"
    prompt = (
        f"Определи, какой из двух вариантов является более правдоподобным {q_text} для следующего утверждения.\n"
        f"Утверждение: {premise}\n"
        f"Вариант 1: {choice1}\n"
        f"Вариант 2: {choice2}\n"
        f"Ответь только номером варианта: 1 или 2."
    )
    messages = [{"role": "user", "content": prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def get_prediction(premise, choice1, choice2, question):
    prompt = build_parus_prompt(premise, choice1, choice2, question)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    if '1' in answer:
        return 0
    elif '2' in answer:
        return 1
    else:
        return None

true_labels = []
pred_labels = []

for example in train_sample:
    premise = example['premise']
    choice1 = example['choice1']
    choice2 = example['choice2']
    question = example['question']
    true_label = example['label']

    pred = get_prediction(premise, choice1, choice2, question)
    true_labels.append(true_label)
    pred_labels.append(pred)

    if len(true_labels) <= 5:
        print(f"Premise: {premise[:60]}...")
        print(f"Q: {question}, Choice1: {choice1[:30]}..., Choice2: {choice2[:30]}...")
        print(f"True: {true_label} -> {'choice1' if true_label==0 else 'choice2'}, Pred: {pred}")
        print("-" * 40)

valid = [i for i, p in enumerate(pred_labels) if p is not None]
true_labels = [true_labels[i] for i in valid]
pred_labels = [pred_labels[i] for i in valid]

if true_labels:
    acc = accuracy_score(true_labels, pred_labels)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary')
    print(f"\n✅ Accuracy на {len(true_labels)} примерах: {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
else:
    print("❌ Нет валидных предсказаний.")

Premise: Женщина подала в суд на преследователя....
Q: cause, Choice1: Мужчина позвонил ей...., Choice2: Мужчина ходил за ней повсюду....
True: 1 -> choice2, Pred: 1
----------------------------------------
Premise: Я пил воду из фонтанчика....
Q: cause, Choice1: У меня была жажда...., Choice2: Меня тошнило....
True: 0 -> choice1, Pred: 1
----------------------------------------
Premise: Власти пообещали сохранить в тайне личность жертвы преступле...
Q: effect, Choice1: Жертва изо всех сил пыталась в..., Choice2: Они скрывали имя жертвы от общ...
True: 1 -> choice2, Pred: 1
----------------------------------------
Premise: Мои ноги были в волдырях....
Q: cause, Choice1: Я ходил в походы...., Choice2: Я ходил купаться....
True: 0 -> choice1, Pred: 1
----------------------------------------
Premise: Я хохотал без остановки....
Q: effect, Choice1: Мои зубы начали стучать...., Choice2: Из глаз брызнули слёзы....
True: 1 -> choice2, Pred: 1
----------------------------------------

✅ Accura